# SLURM from Jupyter: Student Lab

Hands-on lab for learning SLURM from a notebook using the local `slurm_magic.py` file in this folder.

What you will practice:
- load the local SLURM magics extension
- inspect the cluster with `sinfo` and `squeue`
- submit a batch job with `%%sbatch`
- prepare an interactive allocation with `salloc` / `srun`
- switch output modes between raw text and pandas tables

Tip: run the notebook on a SLURM-connected machine. If a command is unavailable on your system, leave the cell unexecuted and read the example.

## 1. Load the local extension

This notebook loads the patched local module instead of the archived `pip` package. That keeps the exercises consistent with the version in this repository.

In [1]:
from IPython import get_ipython
from importlib.util import module_from_spec, spec_from_file_location
from pathlib import Path

ip = get_ipython()
matches = [path for path in Path.cwd().rglob('slurm_magic.py') if 'p4_slurm_notebook' in path.parts]
if not matches:
    raise FileNotFoundError('Could not find local slurm_magic.py in the workspace tree')
module_path = sorted(matches)[0]
spec = spec_from_file_location('local_slurm_magic', module_path)
module = module_from_spec(spec)
spec.loader.exec_module(module)
module.load_ipython_extension(ip)
print(f'Loaded local slurm_magic from {module_path}')

ERROR:tornado.general:Uncaught exception in ZMQStream callback
Traceback (most recent call last):
  File "/net/afscra/people/plgjworek/.venv/lib64/python3.9/site-packages/traitlets/traitlets.py", line 632, in get
    value = obj._trait_values[self.name]
KeyError: '_control_lock'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/net/afscra/people/plgjworek/.venv/lib64/python3.9/site-packages/zmq/eventloop/zmqstream.py", line 565, in _log_error
    f.result()
  File "/net/afscra/people/plgjworek/.venv/lib64/python3.9/site-packages/ipykernel/kernelbase.py", line 301, in dispatch_control
    async with self._control_lock:
  File "/net/afscra/people/plgjworek/.venv/lib64/python3.9/site-packages/traitlets/traitlets.py", line 687, in __get__
    return t.cast(G, self.get(obj, cls))  # the G should encode the Optional
  File "/net/afscra/people/plgjworek/.venv/lib64/python3.9/site-packages/traitlets/traitlets.py", line 649, in get

Loaded local slurm_magic from /net/afscra/people/plgjworek/p4_slurm_notebook/slurm_magic.py


## 2. Warm-up: cluster snapshot

Start with raw output so you can see the exact command response. Then switch to pandas mode and compare the behavior.

Questions to think about:
- Which partitions are visible?
- What does SLURM think about your account or username?
- Do you see nodes, time limits, or queue state?

In [2]:
%slurm display raw
%sinfo -s
%squeue -u $USER

PARTITION       AVAIL  TIMELIMIT   NODES(A/I/O/T) NODELIST
all                up 3-00:00:00   414/350/33/797 ac[0001-0788],ag[0001-0009]
cpu                up 3-00:00:00   412/343/33/788 ac[0001-0788]
cpu-lowprio        up 3-00:00:00   412/343/33/788 ac[0001-0788]
cpu-bigmem         up 3-00:00:00     51/204/1/256 ac[0129-0384]
cpu-lowmem         up 3-00:00:00   361/139/32/532 ac[0001-0128,0385-0788]
gpu                up 3-00:00:00          2/7/0/9 ag[0001-0009]
plgrid*            up 3-00:00:00   412/343/33/788 ac[0001-0788]
plgrid-bigmem      up 3-00:00:00     51/204/1/256 ac[0129-0384]
plgrid-gpu-v100    up 2-00:00:00          2/7/0/9 ag[0001-0009]
plgrid-long        up 7-00:00:00   361/139/32/532 ac[0001-0128,0385-0788]
plgrid-now         up   12:00:00   361/139/32/532 ac[0001-0128,0385-0788]
plgrid-testing     up    1:00:00   361/139/32/532 ac[0001-0128,0385-0788]
             JOBID PARTITION     NAME     USER ST       TIME  NODES NODELIST(REASON)


## 3. Exercise: read the queue as a table

1. Switch to pandas mode.
2. Run `squeue` again.
3. Inspect the columns and note what information is useful for job tracking.

Hint: if you already submitted a job, use its job id to narrow the view.

In [3]:
%slurm display pandas
queue = get_ipython().run_line_magic('squeue', '-u $USER')
queue

,JOBID,PARTITION,NAME,USER,ST,TIME,NODES,NODELIST(REASON)


## 4. Batch jobs with `%%sbatch`

This is the most important pattern for notebook-based SLURM work: write a small batch script inside a cell and submit it directly.

Suggested experiment:
- change the `--job-name`,
- add one more `echo`,
- inspect the output file produced by SLURM.

In [4]:
%%sbatch -N1 -t 00:02:00 --job-name=lab-demo
#!/bin/bash
echo 'Hello from the SLURM notebook lab'
hostname
date
pwd

Submitted batch job 20217820


## 5. Clean up and monitor

Use the standard SLURM tools from inside the notebook to follow the job lifecycle.

Try these after submitting a job:
- `%squeue -u $USER`
- `%scancel <jobid>`
- `%scontrol show job <jobid>`

If `squeue` still shows an error, check whether the username was expanded correctly by the local module.

In [5]:
%slurm display raw
# Replace <jobid> with the number printed by %%sbatch
# %squeue -u $USER
# %scontrol show job <jobid>
# %scancel <jobid>

'raw'

## 6. Interactive allocation pattern

`salloc` gives you resources for an interactive session. In a live cluster session you can combine it with `srun --pty bash -l`.

Do not run this cell unless your cluster supports interactive allocations from the notebook environment. The command is here so you can study the pattern and adapt it to your site rules.

In [ ]:
# Example interactive pattern for a SLURM cluster
# %salloc -N1 -t 00:10:00 -- srun --pty bash -l
# After the allocation starts you can run shell commands on the compute node.
# For notebook workflows, this is often where you start a kernel or launch analysis tools.

## 7. Challenge task

Pick one of the following and try it on your cluster:
- submit a batch job that prints the hostname and GPU information,
- switch `squeue` to pandas mode and inspect the table columns,
- add `scontrol show partition <name>` to learn more about a partition.

Write down one thing you learned about how SLURM behaves from inside Jupyter.

## 8. Takeaways

- The notebook can call SLURM directly through the local `slurm_magic.py` file.
- Raw mode is good for exact command output.
- Pandas mode is useful when the command returns tabular data.
- Batch jobs and interactive allocations are the two core workflows to remember.